# ViGoEmotions Code-Switching Analysis

Notebook này phân tích mức độ **code-switching** (trộn lẫn ngôn ngữ) trong ViGoEmotions trước khi huấn luyện các mô hình.

Trong notebook này, code-switching **chỉ** được định nghĩa là việc chèn ngôn ngữ khác vào câu tiếng Việt (chủ yếu Anh - Việt - Trung). Emoji, emoticon và teencode/tiếng lóng thuần Việt **không** được coi là code-switching.

Mục tiêu:
- Đọc dữ liệu ViGoEmotions từ Hugging Face/Excel/CSV/Pickle.
- Gắn cờ comment có chèn tiếng Anh, tiếng Trung (chữ Hán hoặc phiên âm đã Việt hoá như 谢谢 → "xia xìa"), hoặc ngôn ngữ nước ngoài khác (Hàn/Nhật).
- Tạo nhãn `cs_group`: `pure_vi`, `english_mixed`, `chinese_mixed`, `other_foreign`, `multi_mixed`.
- Thống kê phân bố code-switching theo split và theo emotion label.
- Xuất file CSV để dùng cho báo cáo và để chia subset khi evaluate baseline/model mới.


In [ ]:
import ast
import json
import os
import pickle
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from datasets import load_dataset
except ImportError:
    load_dataset = None

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_columns', 80)


## 1. Cấu hình nguồn dữ liệu

Mặc định notebook tải trực tiếp ViGoEmotions từ Hugging Face bằng `datasets.load_dataset`.

Vì ViGoEmotions là gated dataset, bạn cần được cấp quyền và đã login Hugging Face trước khi chạy:

```bash
huggingface-cli login
```

Nếu muốn dùng file local, đặt `USE_HF_DATASET = False` và sửa `DATA_PATH`.

Notebook hỗ trợ:
- Hugging Face dataset: `uitnlp/vigoemotions`.
- `.xlsx`: có sheet `train`, `val`/`dev`, `test`.
- `.csv`: có cột `text`, `labels`, và có thể có cột `set`/`split`.
- `.pkl`: tuple `(train_df, val_df, test_df)` giống các notebook `_s3`.

In [ ]:
USE_HF_DATASET = True
HF_DATASET_NAME = 'uitnlp/vigoemotions'

# Chỉ dùng khi USE_HF_DATASET = False
DATA_PATH = None

# Nếu chạy local và có file khác, ví dụ:
# DATA_PATH = '/Users/nhule/Desktop/KLTN/ViGoEmotions/corpus/dataset_V1.xlsx'
# DATA_PATH = '/Users/nhule/Desktop/KLTN/ViGoEmotions/corpus/s3_normalized_train-val-test.pkl'

OUT_DIR = 'outputs_code_switching'
os.makedirs(OUT_DIR, exist_ok=True)

HF_DATASET_NAME if USE_HF_DATASET else DATA_PATH

In [ ]:
def normalize_hf_split(split_name):
    name = str(split_name).lower()
    if name in ['validation', 'valid', 'dev']:
        return 'val'
    return name


def load_vigoemotions_from_hf(dataset_name='uitnlp/vigoemotions'):
    if load_dataset is None:
        raise ImportError(
            "Chưa cài package `datasets`. Hãy chạy `pip install datasets` "
            "hoặc đặt USE_HF_DATASET = False để đọc file local."
        )
    try:
        ds = load_dataset(dataset_name)
    except Exception as exc:
        raise RuntimeError(
            "Không tải được ViGoEmotions từ Hugging Face. Nếu dataset gated, hãy kiểm tra bạn đã được cấp quyền "
            "và đã login bằng `huggingface-cli login`. Lỗi gốc: " + str(exc)
        ) from exc

    frames = []
    for split_name, split_data in ds.items():
        part = split_data.to_pandas()
        part['split'] = normalize_hf_split(split_name)
        frames.append(part)

    df = pd.concat(frames, ignore_index=True)
    if 'label' in df.columns and 'labels' not in df.columns:
        df = df.rename(columns={'label': 'labels'})
    return df


def load_vigoemotions_from_file(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ['.xlsx', '.xls']:
        xls = pd.ExcelFile(path)
        sheet_map = {s.lower(): s for s in xls.sheet_names}
        train_df = pd.read_excel(path, sheet_name=sheet_map.get('train', xls.sheet_names[0]))
        val_name = sheet_map.get('val') or sheet_map.get('dev') or sheet_map.get('validation')
        test_name = sheet_map.get('test')
        val_df = pd.read_excel(path, sheet_name=val_name) if val_name else pd.DataFrame()
        test_df = pd.read_excel(path, sheet_name=test_name) if test_name else pd.DataFrame()
        train_df['split'] = 'train'
        val_df['split'] = 'val'
        test_df['split'] = 'test'
        return pd.concat([train_df, val_df, test_df], ignore_index=True)

    if ext == '.csv':
        df = pd.read_csv(path)
        if 'split' not in df.columns and 'set' in df.columns:
            df = df.rename(columns={'set': 'split'})
        if 'split' not in df.columns:
            df['split'] = 'unknown'
        return df

    if ext in ['.pkl', '.pickle']:
        with open(path, 'rb') as f:
            obj = pickle.load(f)
        if isinstance(obj, tuple) and len(obj) == 3:
            train_df, val_df, test_df = obj
            train_df = train_df.copy(); val_df = val_df.copy(); test_df = test_df.copy()
            train_df['split'] = 'train'; val_df['split'] = 'val'; test_df['split'] = 'test'
            return pd.concat([train_df, val_df, test_df], ignore_index=True)
        if isinstance(obj, pd.DataFrame):
            df = obj.copy()
            if 'split' not in df.columns:
                df['split'] = 'unknown'
            return df
        raise ValueError('Unsupported pickle format')

    raise ValueError(f'Unsupported file extension: {ext}')


if USE_HF_DATASET:
    df = load_vigoemotions_from_hf(HF_DATASET_NAME)
else:
    if DATA_PATH is None:
        raise ValueError('DATA_PATH must be set when USE_HF_DATASET = False')
    df = load_vigoemotions_from_file(DATA_PATH)

df.head()

In [ ]:
print(df.shape)
print(df.columns.tolist())
display(df['split'].value_counts(dropna=False))
display(df[['split', 'text', 'labels']].head())

## 2. Detector code-switching dạng rule-based

Detector nhận diện việc trộn ngôn ngữ khác vào tiếng Việt, gồm 3 nguồn:

- **Tiếng Anh**: token ASCII là từ tiếng Anh phổ biến hoặc có hậu tố tiếng Anh, và không phải viết tắt/teencode tiếng Việt.
- **Tiếng Trung**: chữ Hán trực tiếp, hoặc phiên âm đã Việt hoá được liệt kê trong `ZH_TRANSLIT_SEQS` (ví dụ 谢谢 → "xia xìa", 你好 → "nỉ hảo").
- **Ngôn ngữ khác**: các hệ chữ nước ngoài khác như Hangul (Hàn), Kana (Nhật).

Teencode/tiếng lóng thuần Việt và emoji chỉ được giữ lại làm metadata mô tả, **không** tính là code-switching. Đây là detector thực nghiệm, không phải gold annotation; bạn nên kiểm tra thủ công một số mẫu ở phần 6.


In [ ]:
import re
import unicodedata

# ===========================================================================
# Rule-based CROSS-LANGUAGE code-switching detector
# ---------------------------------------------------------------------------
# Ở đây code-switching CHỈ gồm việc trộn ngôn ngữ khác vào tiếng Việt
# (chủ yếu Anh - Việt - Trung). Emoji, emoticon và teencode/tiếng lóng thuần
# Việt KHÔNG được tính là code-switching (teencode chỉ dùng để loại trừ, tránh
# nhận nhầm là tiếng Anh). Từ nước ngoài đã được "Việt hoá" / phiên âm (ví dụ
# tiếng Trung 谢谢 -> "xia xìa", 你好 -> "nỉ hảo") vẫn được tính là
# code-switching thông qua ZH_TRANSLIT_SEQS bên dưới.
# ===========================================================================

URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'(?<!\w)@[\w_]+')
HASHTAG_RE = re.compile(r'(?<!\w)#[\w_]+')
# Token chữ Latin (gồm cả dấu tiếng Việt)
WORD_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]+(?:[-'][A-Za-zÀ-ỹĐđ]+)?", re.UNICODE)
ASCII_ALPHA_RE = re.compile(r'^[A-Za-z]+$')
VIET_CHARS_RE = re.compile(r'[ăâđêôơưáàảãạấầẩẫậắằẳẵặéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ]', re.IGNORECASE)
EMOJI_RE = re.compile('[\U0001F300-\U0001FAFF\U00002700-\U000027BF\U00002600-\U000026FF]+')

# Hệ chữ nước ngoài không nhập nhằng
HAN_RE = re.compile('[㐀-䶿一-鿿豈-﫿]+')   # chữ Hán (Trung/Nhật kanji)
HANGUL_RE = re.compile('[가-힣]+')                          # Hangul (Hàn)
KANA_RE = re.compile('[぀-ヿ]+')                            # Kana (Nhật)

# Tiếng Việt (đã bỏ dấu) + teencode thuần Việt -> KHÔNG coi là tiếng nước ngoài.
# Chỉ dùng để loại trừ, giúp is_english_like không bắt nhầm teencode thành tiếng Anh.
VI_EXCLUDE = set('''
toi tui tao may ban minh chung nguoi la thi ma va hay nhung vi voi cho cua trong tren duoi nay kia do day roi chua
khong ko k k0 kh khum hong hok hem duoc dc dk dx di den ve an uong lam hoc choi xem nghe noi nghi biet thay ghe qua
troi rat hoi luon nua nha nhe dang da se mot hai ba bon nam sau bay tam chin muoi
j z dz zj zi v vs vl vc vcl vkl vãi vai cx cg cung chu chz dm dcm cmt cmm clm mik mk mjk mn mng ae ny iu yeu lun
rui ruoi bt bth bthuong thik ib rep kb tks pls plz wa qa qá bn b t m r ui z0 hj hjhj kk kkk haha hihi hehe huhu
'''.split())

# Từ tiếng Anh phổ biến (gồm cả loanword giới trẻ hay chèn vào câu tiếng Việt).
EN_COMMON = set('''
love like hate happy sad angry fear sorry thanks thank please ok okay yes no good bad best worst cute nice cool hot
trend trending fan idol team movie music video clip live stream comment share post status story drama fake real toxic
cringe brainwash anti respect support follow unfollow block game gaming win lose loser boy girl baby bro sis friend
family crush deadline job work school class online offline ship shipper couple hello bye view sub subscribe show style
admin facebook youtube tiktok perfect legend wow king queen
'''.split())

EN_SUFFIX_RE = re.compile(r'(ing|tion|ment|ness|ous|ive|er|ed|ly|ship|ful)$')

# Phiên âm tiếng Trung hay gặp trong bình luận tiếng Việt (đã bỏ dấu; cho phép
# viết liền hoặc cách khoảng). Thêm cụm mới vào đây khi cần mở rộng.
ZH_TRANSLIT_SEQS = [
    ['xie', 'xie'], ['xia', 'xia'],            # 谢谢  cảm ơn
    ['ni', 'hao'], ['nin', 'hao'],             # 你好  xin chào
    ['da', 'jia', 'hao'],                      # 大家好 chào mọi người
    ['wo', 'ai', 'ni'],                        # 我爱你 anh yêu em
    ['zai', 'jian'],                           # 再见  tạm biệt
    ['dui', 'bu', 'qi'],                       # 对不起 xin lỗi
    ['mei', 'guan', 'xi'],                     # 没关系 không sao
    ['jia', 'you'],                            # 加油  cố lên
    ['bao', 'bei'],                            # 宝贝  cục cưng
    ['gan', 'bei'],                            # 干杯  cạn ly
    ['piao', 'liang'],                         # 漂亮  xinh đẹp
    ['shuai', 'ge'],                           # 帅哥  anh đẹp trai
    ['mei', 'nu'],                             # 美女  gái xinh
    ['shen', 'me'],                            # 什么  cái gì
    ['wei', 'shen', 'me'],                     # 为什么 tại sao
    ['zhen', 'de'],                            # 真的  thật sự
    ['gong', 'xi', 'fa', 'cai'],               # 恭喜发财 chúc mừng năm mới
    ['wan', 'an'],                             # 晚安  ngủ ngon
    ['zao', 'an'],                             # 早安  chào buổi sáng
    ['wo', 'men'], ['ni', 'men'],              # 我们/你们 chúng ta/các bạn
    ['tai', 'hao', 'le'],                      # 太好了 tuyệt quá
]
ZH_TRANSLIT_PATTERNS = [re.compile(r'\b' + r'\s*'.join(seq) + r'\b') for seq in ZH_TRANSLIT_SEQS]
ZH_TRANSLIT_LABELS = [' '.join(seq) for seq in ZH_TRANSLIT_SEQS]


def fold_accents(text):
    text = unicodedata.normalize('NFD', str(text))
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return text.replace('Đ', 'D').replace('đ', 'd').lower()


def is_english_like(token):
    t = token.lower()
    if not ASCII_ALPHA_RE.match(token):
        return False
    if len(t) <= 2:
        return False
    if t in EN_COMMON:
        return True
    if t in VI_EXCLUDE:
        return False
    return bool(EN_SUFFIX_RE.search(t))


def detect_chinese(raw, folded):
    han_tokens = HAN_RE.findall(raw)
    translit = [lab for pat, lab in zip(ZH_TRANSLIT_PATTERNS, ZH_TRANSLIT_LABELS) if pat.search(folded)]
    return han_tokens + translit


def analyze_text(text):
    raw = str(text)
    folded = fold_accents(raw)
    word_tokens = WORD_RE.findall(raw)
    num_word_tokens = len(word_tokens)

    english_tokens = [tok for tok in word_tokens if is_english_like(tok)]
    chinese_tokens = detect_chinese(raw, folded)
    other_foreign_tokens = HANGUL_RE.findall(raw) + KANA_RE.findall(raw)

    has_english = len(english_tokens) > 0
    has_chinese = len(chinese_tokens) > 0
    has_other_foreign = len(other_foreign_tokens) > 0

    langs = []
    if has_english:
        langs.append('en')
    if has_chinese:
        langs.append('zh')
    if has_other_foreign:
        langs.append('other')

    if not langs:
        cs_group = 'pure_vi'
    elif len(langs) >= 2:
        cs_group = 'multi_mixed'
    elif langs[0] == 'en':
        cs_group = 'english_mixed'
    elif langs[0] == 'zh':
        cs_group = 'chinese_mixed'
    else:
        cs_group = 'other_foreign'

    return pd.Series({
        'num_tokens': num_word_tokens,
        'has_vietnamese_diacritic': bool(VIET_CHARS_RE.search(raw)),
        'has_english': has_english,
        'english_count': len(english_tokens),
        'english_ratio': len(english_tokens) / max(num_word_tokens, 1),
        'english_tokens': ', '.join(english_tokens[:20]),
        'has_chinese': has_chinese,
        'chinese_count': len(chinese_tokens),
        'chinese_tokens': ', '.join(chinese_tokens[:20]),
        'has_other_foreign': has_other_foreign,
        'other_foreign_tokens': ', '.join(other_foreign_tokens[:20]),
        'foreign_langs': ', '.join(langs),
        'emoji_count': len(EMOJI_RE.findall(raw)),
        'url_count': len(URL_RE.findall(raw)),
        'mention_count': len(MENTION_RE.findall(raw)),
        'hashtag_count': len(HASHTAG_RE.findall(raw)),
        'is_code_switched': len(langs) > 0,
        'cs_group': cs_group,
    })


import ast


def parse_labels(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    s = str(value)
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass
    return [x.strip() for x in s.strip('[]').replace('"', '').replace("'", '').split(',') if x.strip()]


In [ ]:
analysis_df = pd.concat([df.reset_index(drop=True), df['text'].apply(analyze_text)], axis=1)
analysis_df['label_list'] = analysis_df['labels'].apply(parse_labels)

display(analysis_df[['split', 'text', 'labels', 'cs_group', 'foreign_langs', 'english_tokens', 'chinese_tokens']].head(20))
analysis_df.to_csv(os.path.join(OUT_DIR, 'vigo_code_switching_annotations.csv'), index=False)
print('Saved:', os.path.join(OUT_DIR, 'vigo_code_switching_annotations.csv'))

## 3. Thống kê tổng quan

In [ ]:
summary = (
    analysis_df.groupby('split')
    .agg(
        n=('text', 'size'),
        code_switched=('is_code_switched', 'sum'),
        has_english=('has_english', 'sum'),
        has_chinese=('has_chinese', 'sum'),
        has_other_foreign=('has_other_foreign', 'sum'),
        avg_english_ratio=('english_ratio', 'mean'),
    )
    .reset_index()
)
for col in ['code_switched', 'has_english', 'has_chinese', 'has_other_foreign']:
    summary[col + '_pct'] = summary[col] / summary['n'] * 100

display(summary)
summary.to_csv(os.path.join(OUT_DIR, 'summary_by_split.csv'), index=False)


In [ ]:
group_counts = pd.crosstab(analysis_df['split'], analysis_df['cs_group'], margins=True)
display(group_counts)
group_counts.to_csv(os.path.join(OUT_DIR, 'cs_group_by_split.csv'))

ax = group_counts.drop(index='All', errors='ignore').drop(columns='All', errors='ignore').plot(kind='bar', stacked=True, figsize=(10, 5))
ax.set_title('Code-switching groups by split')
ax.set_xlabel('Split')
ax.set_ylabel('Number of comments')
plt.tight_layout()
plt.show()

## 4. Top English/Chinese tokens

In [ ]:
def split_token_string(s):
    if pd.isna(s) or not str(s).strip():
        return []
    return [x.strip().lower() for x in str(s).split(',') if x.strip()]

english_counter = Counter(tok for s in analysis_df['english_tokens'] for tok in split_token_string(s))
chinese_counter = Counter(tok for s in analysis_df['chinese_tokens'] for tok in split_token_string(s))

top_english = pd.DataFrame(english_counter.most_common(50), columns=['token', 'count'])
top_chinese = pd.DataFrame(chinese_counter.most_common(50), columns=['token', 'count'])

display(top_english.head(30))
display(top_chinese.head(30))

top_english.to_csv(os.path.join(OUT_DIR, 'top_english_tokens.csv'), index=False)
top_chinese.to_csv(os.path.join(OUT_DIR, 'top_chinese_tokens.csv'), index=False)


## 5. Phân bố code-switching theo label

Nếu `labels` đang là số, bảng sẽ theo id label. Nếu muốn tên label, nạp thêm `label_dict.json` và map lại.

In [ ]:
exploded = analysis_df.explode('label_list').copy()
exploded['label_list'] = exploded['label_list'].astype(str)

label_cs = (
    exploded.groupby('label_list')
    .agg(
        n=('text', 'size'),
        code_switched=('is_code_switched', 'sum'),
        english_mixed=('has_english', 'sum'),
        chinese_mixed=('has_chinese', 'sum'),
    )
    .reset_index()
)
label_cs['code_switched_pct'] = label_cs['code_switched'] / label_cs['n'] * 100
label_cs = label_cs.sort_values(['code_switched_pct', 'n'], ascending=[False, False])

display(label_cs.head(40))
label_cs.to_csv(os.path.join(OUT_DIR, 'code_switching_by_label.csv'), index=False)


## 6. Lấy mẫu để kiểm tra thủ công

Bước này quan trọng: rule-based detector có thể sai. Bạn nên kiểm tra khoảng 50-100 mẫu để điều chỉnh `EN_COMMON`, `VI_STOPWORDS_ASCII`, `TEENCODE`.

In [ ]:
sample_cols = ['split', 'text', 'labels', 'cs_group', 'foreign_langs', 'english_tokens', 'chinese_tokens']

for group in ['english_mixed', 'chinese_mixed', 'multi_mixed', 'other_foreign', 'pure_vi']:
    subset = analysis_df[analysis_df['cs_group'] == group]
    if len(subset) == 0:
        continue
    print('\n###', group)
    display(subset[sample_cols].sample(min(10, len(subset)), random_state=42))

manual_parts = []
for _, group_df in analysis_df.groupby('cs_group'):
    manual_parts.append(group_df.sample(min(30, len(group_df)), random_state=42))
manual_sample = pd.concat(manual_parts, ignore_index=True)[sample_cols]
manual_sample.to_csv(os.path.join(OUT_DIR, 'manual_check_samples.csv'), index=False)
print('Saved:', os.path.join(OUT_DIR, 'manual_check_samples.csv'))

## 7. Gợi ý dùng kết quả cho thí nghiệm model

Sau khi chạy notebook này, dùng `vigo_code_switching_annotations.csv` để evaluate model theo subset:

- `all`: toàn bộ test set.
- `pure_vi`: `cs_group == 'pure_vi'`.
- `code_switched`: `is_code_switched == True`.
- `english_mixed`: `cs_group == 'english_mixed'`.
- `chinese_mixed`: `cs_group == 'chinese_mixed'`.

Bảng nên báo cáo trong luận văn:

| Model | All Macro F1 | Pure VI Macro F1 | Code-switched Macro F1 | English-mixed Macro F1 | Chinese-mixed Macro F1 |
|---|---:|---:|---:|---:|---:|
| XLM-R | | | | | |
| PhoBERT | | | | | |
| XLM-R + xLSTM | | | | | |
| PhoBERT + xLSTM | | | | | |
